# 🔵 MODELO 2: LSTM - Captura de Secuencias

## 📋 Objetivo del ejercicio
Mejorar el modelo baseline añadiendo **LSTM** para capturar el orden de las palabras.

## 🎯 Arquitectura
- **Embedding**: Convierte palabras en vectores
- **LSTM**: Procesa secuencias en orden, con memoria
- **Dense**: Capa de clasificación

## ✅ Mejora sobre Modelo 1
LSTM captura **dependencias temporales** y el **orden** de las palabras.

### 📚 Teoría de examen:
> "Las RNN/LSTM son necesarias cuando el orden temporal importa.
> En NLP, el orden cambia el significado: 'not spam' ≠ 'spam not'"

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)

In [ ]:
# Leer datos
train = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/train.csv", index_col="row_id")
test = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/test.csv", index_col="row_id")

print(f"📊 Train shape: {train.shape}")
print(f"📊 Test shape: {test.shape}")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

tf.random.set_seed(seed)

X = train['message'].values
y = train['spam_label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

In [ ]:
# Parámetros - Aumentamos embedding_dim para LSTM
MAX_WORDS = 5000
MAX_LEN = 100
EMBEDDING_DIM = 64  # ⬆️ Más dimensiones para capturar más información

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(test['message'].values)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post')

print(f"✅ Datos listos para LSTM")

## 🏗️ Modelo con LSTM

### 📚 ¿Qué es LSTM?
- **Long Short-Term Memory**: RNN con memoria a largo plazo
- **4 gates**: input, forget, output, cell state
- **Ventaja**: Captura dependencias temporales largas
- **Problema del vanishing gradient**: Resuelto con gates

### 🔑 Pregunta de examen típica:
**P: ¿Por qué usar LSTM en lugar de GlobalAveragePooling?**

**R: Porque LSTM procesa secuencias EN ORDEN y mantiene memoria del contexto previo. 
GlobalAveragePooling solo promedia, perdiendo el orden.**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(64),  # ✨ MEJORA: Captura orden y contexto
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### 📊 Nota sobre parámetros:
Observa que LSTM tiene **muchos más parámetros** que el modelo baseline.
Esto es porque tiene 4 gates (input, forget, output, cell).

In [ ]:
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

In [ ]:
val_loss, val_acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 Validation Accuracy: {val_acc:.4f}")
print(f"📊 Validation Loss: {val_loss:.4f}")
print(f"\n✅ Esperamos MEJOR accuracy que el baseline (~5-10% más)")

In [ ]:
# Predicciones
y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

submission = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/sample_submission.csv")
submission["spam_label"] = y_pred
submission.to_csv('submission_lstm.csv', index=False)

print("✅ Submission creado: submission_lstm.csv")
submission.head()

---

## 📚 Resumen para Examen

### ✅ Ventajas LSTM:
- **Captura orden temporal** de las palabras
- **Memoria a largo plazo** (cell state)
- Mejor para secuencias con contexto

### ❌ Desventaja:
- **Más lento** que baseline
- **Más parámetros** → necesita más datos
- **Puede sufrir overfitting** si no se regulariza

### 🔄 Siguiente paso:
**Añadir Dropout** para prevenir overfitting.